# mpH2MM / smFRET Analysis — CSV burst files (Alexa555 / Alexa647)

**Input:** Pre-analysed burst summary CSV merged from multiple PTU files.  
**Donor:** Alexa 555 | **Acceptor:** Alexa 647

**CSV columns used:**
- `n_DD` — Dex-Dem photons (donor channel, donor excitation)
- `n_DA` — Dex-Aem photons (acceptor channel, donor excitation)
- `n_AA` — Aex-Aem photons (acceptor channel, acceptor excitation)
- `E_app`, `S_app` — apparent FRET efficiency and stoichiometry
- `time_length` — burst duration in seconds
- `filename` — source PTU file identifier

**Pipeline:**
1. Load and inspect CSV
2. Quality filters (S filter, minimum photon counts)
3. ES jointplot
4. First-round H²MM — identify and remove low-FRET bursts
5. Second-round H²MM on FRET-active bursts
6. Dwell-time extraction and survival plots
7. BVA (Burst Variance Analysis)
8. Per-file consistency check

**Environment:** Python 3.10, numpy 1.26.4, burstH2MM 0.1.7, H2MM_C 2.1.2

**Index convention:** models list is 0-based — `models[0]`=1-state, `models[1]`=2-state, etc.

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import H2MM_C as h2

print('numpy     :', np.__version__)
print('pandas    :', pd.__version__)
print('H2MM_C    :', h2.__version__)

## 2 · Helper functions

In [ ]:
def es_jointplot(E, S, title='', bins=30):
    """FRETbursts-style ES jointplot from E and S arrays."""
    fig = plt.figure(figsize=(6, 5))
    gs  = gridspec.GridSpec(2, 2, width_ratios=[3,1], height_ratios=[1,3],
                            hspace=0.05, wspace=0.05)
    ax_joint = fig.add_subplot(gs[1, 0])
    ax_top   = fig.add_subplot(gs[0, 0], sharex=ax_joint)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.hist2d(E, S, bins=bins, range=((0,1),(0,1)), cmap='Blues')
    ax_top.hist(E, bins=bins, range=(0,1), color='steelblue')
    ax_right.hist(S, bins=bins, range=(0,1), orientation='horizontal',
                  color='steelblue')

    ax_joint.set_xlabel(r'$E_{app}$')
    ax_joint.set_ylabel(r'$S_{app}$')
    ax_joint.set_xlim(0,1); ax_joint.set_ylim(0,1)
    ax_top.set_ylabel('# Bursts')
    ax_right.set_xlabel('# Bursts')
    plt.setp(ax_top.get_xticklabels(), visible=False)
    plt.setp(ax_right.get_yticklabels(), visible=False)
    ax_joint.text(0.97, 0.97, f'# Bursts: {len(E)}',
                  transform=ax_joint.transAxes,
                  ha='right', va='top', fontsize=9)
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
def build_h2mm_streams(df, mean_interval_ticks=100):
    """
    Build H2MM_C photon stream arrays from burst summary CSV.

    Each burst → detector index array (0=donor, 1=acceptor) + timestamp array.
    Uses Poisson-distributed inter-photon times to avoid H2MM_C numerical
    issues caused by equal-spaced synthetic timestamps.

    Only Dex photons (n_DD + n_DA) are used — standard for 2-colour H2MM.
    """
    indexes, times = [], []
    rng = np.random.default_rng(seed=42)
    for _, row in df.iterrows():
        nDD     = int(row['n_DD'])
        nDA     = int(row['n_DA'])
        n_total = nDD + nDA
        if n_total < 2:
            continue
        det = np.array([0]*nDD + [1]*nDA, dtype=np.uint8)
        rng.shuffle(det)
        ipt = rng.poisson(mean_interval_ticks, size=n_total).astype(np.uint64)
        ipt[ipt == 0] = 1
        ts  = np.cumsum(ipt)
        indexes.append(det)
        times.append(ts)
    return indexes, times

In [ ]:
def fit_h2mm(indexes, times, max_state=6, max_iter=3600):
    """
    Fit H2MM models for 1 to max_state states using EM_H2MM_C.
    Returns list of h2mm_model objects (0-indexed: [0]=1-state, [1]=2-state...).
    """
    models = []
    print(f'Fitting {max_state} models...')
    for n in range(1, max_state + 1):
        init   = h2.factory_h2mm_model(n, 2)
        result = h2.EM_H2MM_C(init, indexes, times, max_iter=max_iter)
        models.append(result)
        try:
            e_vals = np.round(result.obs[:, 1], 3)
        except Exception:
            e_vals = ['?'] * n
        print(f'  {n}-state: loglik={result.loglik:.2f}  '
              f'BIC={result.bic:.2f}  '
              f'niter={result.niter}  '
              f'conv={result.conv_str}  '
              f'E={e_vals}')
    print('Done.')
    return models

In [ ]:
def plot_model_selection(models):
    """
    Plot BIC model selection.
    Pick the model at the BIC elbow — where improvement becomes negligible.
    Models that hit max_iter (not converged) are flagged.
    """
    n_states_list = list(range(1, len(models)+1))
    bics  = [m.bic for m in models]
    convd = [m.conv_str == 'final optimized model by converged' for m in models]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(n_states_list, bics, 'o-', color='steelblue')
    # Mark non-converged models
    for i, (n, bic, conv) in enumerate(zip(n_states_list, bics, convd)):
        if not conv:
            ax.plot(n, bic, 'rx', markersize=12, markeredgewidth=2,
                    label='not converged' if i == next(
                        j for j, c in enumerate(convd) if not c) else '')
    ax.set_xlabel('Number of states')
    ax.set_ylabel('BIC')
    ax.set_title('BIC model selection — pick elbow; ignore ✗ (not converged)')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # BIC differences to help identify elbow
    print('BIC drops between successive models:')
    for i in range(1, len(bics)):
        drop = bics[i-1] - bics[i]
        flag = '← elbow?' if drop < 1000 else ''
        conv = '✓' if convd[i] else '✗ not converged'
        print(f'  {i}→{i+1} states: ΔBIC={drop:.1f}  {conv}  {flag}')

    # Best = largest drop that is still converged
    best = int(np.argmin(bics))
    print(f'\nLowest BIC: {best+1} states (index {best})')
    return best

In [ ]:
def plot_es_h2mm(model, df, indexes, times, title='H2MM ES plot'):
    """
    ES scatter coloured by dominant Viterbi state per burst.
    """
    E = df['E_app'].values
    S = df['S_app'].values

    paths, _, _, _ = h2.viterbi_path(model, indexes, times)
    dominant_state = np.array([np.bincount(p).argmax() for p in paths])

    n_states = model.nstate
    colors   = plt.cm.tab10(np.linspace(0, 0.9, n_states))

    fig, ax = plt.subplots(figsize=(5, 4))
    for s in range(n_states):
        mask = dominant_state == s
        ax.scatter(E[mask], S[mask], s=4, alpha=0.4, color=colors[s],
                   label=f'State {s}  E≈{model.obs[s,1]:.2f}')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel(r'$E_{app}$'); ax.set_ylabel(r'$S_{app}$')
    ax.legend(markerscale=3, fontsize=8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return dominant_state

In [ ]:
def dwell_survival_plots(model, indexes, times, mean_ph_rate):
    """
    Extract dwell times via Viterbi and plot survival functions per state.
    Dwell durations (photons) are converted to ms using mean_ph_rate.
    """
    paths, _, _, _ = h2.viterbi_path(model, indexes, times)
    n_states       = model.nstate

    dwell_times = [[] for _ in range(n_states)]
    for path in paths:
        if len(path) < 2:
            continue
        transitions = np.where(np.diff(path) != 0)[0] + 1
        boundaries  = np.concatenate([[0], transitions, [len(path)]])
        for start, stop in zip(boundaries[:-1], boundaries[1:]):
            dwell_times[int(path[start])].append(int(stop - start))

    print(f'Total dwells: {sum(len(d) for d in dwell_times)}')
    for s in range(n_states):
        dt_ms = np.array(dwell_times[s]) / mean_ph_rate * 1000
        print(f'  State {s} (E≈{model.obs[s,1]:.2f}): '
              f'{len(dt_ms)} dwells  '
              f'mean={dt_ms[dt_ms>0].mean():.3f} ms  '
              f'max={dt_ms.max():.3f} ms')

    fig, axes = plt.subplots(1, n_states, figsize=(4*n_states, 4), sharey=True)
    if n_states == 1:
        axes = [axes]
    for s, ax in enumerate(axes):
        dt   = np.array(dwell_times[s])
        dt   = dt[dt > 0]
        t_ms = np.sort(dt / mean_ph_rate * 1000)
        surv = 1 - np.arange(1, len(t_ms)+1) / len(t_ms)
        ax.semilogy(t_ms, surv, '.', markersize=3)
        ax.set_xlabel('Dwell time (ms)')
        ax.set_ylabel('Survival probability')
        ax.set_title(f'State {s}  E≈{model.obs[s,1]:.2f}  (n={len(dt)})')
    plt.suptitle('Dwell-time survival functions per H²MM state')
    plt.tight_layout()
    plt.show()
    return dwell_times

In [ ]:
def bva_from_counts(df, chunk_size=5):
    """
    BVA from photon counts (n_DD, n_DA).
    Splits each burst into sub-bursts of chunk_size Dex photons.
    """
    E_list, std_list = [], []
    rng = np.random.default_rng(seed=42)
    for _, row in df.iterrows():
        nDD     = int(row['n_DD'])
        nDA     = int(row['n_DA'])
        n_total = nDD + nDA
        if n_total < chunk_size * 2:
            continue
        det = np.array([0]*nDD + [1]*nDA, dtype=np.uint8)
        rng.shuffle(det)
        Esub = [det[nb:ne].mean()
                for nb, ne in zip(range(0, n_total, chunk_size),
                                  range(chunk_size, n_total+1, chunk_size))]
        if len(Esub) < 2:
            continue
        E_list.append(np.mean(Esub))
        std_list.append(np.std(Esub))
    return np.array(E_list), np.array(std_list)


def bin_bva(E, std, R=10, B_thr=20):
    bn      = np.linspace(0, 1, R+1)
    std_avg = np.full(R, -1.0)
    E_avg   = np.full(R, -1.0)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E) & (E < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std[mask])
            E_avg[i]   = np.mean(E[mask])
    return E_avg, std_avg

n_bva = 5
x_T   = np.arange(0, 1.01, 0.01)
y_T   = np.sqrt((x_T * (1 - x_T)) / n_bva)

## 3 · Load and inspect CSV

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
csv_file = r'C:/path/to/your_merged_bursts.csv'   # ← change this
# ─────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(csv_file)
print(f'Loaded {len(df_raw)} bursts')
print(f'Columns      : {list(df_raw.columns)}')
print(f'Source files : {df_raw["filename"].nunique()} PTU files')
print()
print(df_raw[['n_DD','n_DA','n_AA','E_app','S_app','n_photons']].describe().round(2))

es_jointplot(df_raw['E_app'].values, df_raw['S_app'].values,
             title='All bursts — no filter')

## 4 · Quality filters

- **S filter (0.25–0.75):** removes donor-only (S→1) and acceptor-only (S→0)
- **Min photon counts:** removes small bursts with unreliable E

Adjust thresholds as needed.

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
S_min    = 0.25
S_max    = 0.75
min_n_DD = 5
min_n_DA = 5
min_n_AA = 10
# ─────────────────────────────────────────────────────────────────────────

mask_S  = (df_raw['S_app'] >= S_min) & (df_raw['S_app'] <= S_max)
mask_ph = ((df_raw['n_DD'] >= min_n_DD) &
           (df_raw['n_DA'] >= min_n_DA) &
           (df_raw['n_AA'] >= min_n_AA))

df = df_raw[mask_S & mask_ph].copy().reset_index(drop=True)

print(f'Bursts before filter : {len(df_raw)}')
print(f'After S filter       : {mask_S.sum()}')
print(f'After photon filter  : {(mask_S & mask_ph).sum()}')
print(f'Final dataset        : {len(df)} bursts')

es_jointplot(df['E_app'].values, df['S_app'].values,
             title='After S filter + min photon counts')

## 5 · Build H2MM photon streams

Converts burst photon counts (n_DD, n_DA) into detector index sequences
with Poisson-distributed inter-photon times for H2MM_C.

In [ ]:
indexes, times = build_h2mm_streams(df)
print(f'Streams built    : {len(indexes)} bursts')
print(f'Example burst    : {len(indexes[0])} photons')

total_ph     = df['n_DD'].sum() + df['n_DA'].sum()
total_dur    = df['time_length'].sum()
mean_ph_rate = total_ph / total_dur
print(f'Mean in-burst photon rate: {mean_ph_rate:.0f} ph/s')
print(f'Total burst duration     : {total_dur*1000:.1f} ms')

## 6 · First-round H²MM

Fits 1–6 state models. Use BIC plot to pick the optimal number.
Look for the **elbow** — where ΔBIC drops below ~1000.
Ignore models marked ✗ (hit max iterations, not fully converged).

In [ ]:
models_r1 = fit_h2mm(indexes, times, max_state=6, max_iter=3600)

In [ ]:
best_idx_r1 = plot_model_selection(models_r1)

In [ ]:
# ── SET based on BIC elbow above ──────────────────────────────────────────
# Only use converged models (✓). 0-indexed: 0=1-state, 1=2-state, 2=3-state...
round1_idx = best_idx_r1   # ← override manually if needed, e.g. round1_idx = 2
# ─────────────────────────────────────────────────────────────────────────

model_r1 = models_r1[round1_idx]
print(f'First-round model : {model_r1.nstate} states')
print(f'E per state       : {np.round(model_r1.obs[:,1], 3)}')
print(f'Convergence       : {model_r1.conv_str}')

dominant_state_r1 = plot_es_h2mm(
    model_r1, df, indexes, times,
    title=f'First-round H2MM — {model_r1.nstate} states'
)

## 7 · Remove low-FRET bursts → FRET-active dataset

The lowest-E state corresponds to donor-only or very low-FRET bursts.
These are removed before the second-round fit.

In [ ]:
# Lowest-E state = donor-only / low-FRET
low_fret_state = int(np.argmin(model_r1.obs[:, 1]))
print(f'Low-FRET state  : {low_fret_state}  '
      f'E={model_r1.obs[low_fret_state,1]:.2f}')

fret_mask = dominant_state_r1 != low_fret_state
print(f'Bursts before filter : {len(fret_mask)}')
print(f'Bursts after  filter : {fret_mask.sum()}')

df_fret              = df[fret_mask].reset_index(drop=True)
idx_fret, t_fret     = build_h2mm_streams(df_fret)
total_ph_fret        = df_fret['n_DD'].sum() + df_fret['n_DA'].sum()
mean_ph_rate_fret    = total_ph_fret / df_fret['time_length'].sum()
print(f'Mean ph rate (FRET bursts): {mean_ph_rate_fret:.0f} ph/s')

es_jointplot(df_fret['E_app'].values, df_fret['S_app'].values,
             title='FRET-active bursts after first-round H2MM filter')

## 8 · Second-round H²MM (on FRET-active bursts)

In [ ]:
models_r2 = fit_h2mm(idx_fret, t_fret, max_state=6, max_iter=3600)

In [ ]:
best_idx_r2 = plot_model_selection(models_r2)

In [ ]:
# ── SET based on second-round BIC elbow ───────────────────────────────────
final_idx = best_idx_r2   # ← override manually if needed, e.g. final_idx = 2
# ─────────────────────────────────────────────────────────────────────────

model_final = models_r2[final_idx]
print(f'Final model  : {model_final.nstate} states')
print(f'E per state  : {np.round(model_final.obs[:,1], 3)}')
print(f'Convergence  : {model_final.conv_str}')
print(f'Trans matrix :\n{model_final.trans}')

dominant_state_final = plot_es_h2mm(
    model_final, df_fret, idx_fret, t_fret,
    title=f'Final H2MM — {model_final.nstate} states'
)

## 9 · Dwell-time extraction

In [ ]:
dwell_times = dwell_survival_plots(
    model_final, idx_fret, t_fret,
    mean_ph_rate=mean_ph_rate_fret
)

## 10 · BVA (Burst Variance Analysis)

Points above the shot-noise line (dashed) confirm genuine dynamics.

In [ ]:
E_bva, std_bva = bva_from_counts(df_fret, chunk_size=n_bva)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=20)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(E_bva, std_bva, s=4, alpha=0.3, color='grey', label='per burst')
valid = E_avg > 0
ax.scatter(E_avg[valid], std_avg[valid], s=60, color='blue',
           zorder=5, label='binned avg')
ax.plot(x_T, y_T, 'k--', label='shot noise')
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.set_title('BVA — points above dashed line = real dynamics')
ax.legend()
plt.tight_layout()
plt.show()

## 11 · Per-file consistency check

Checks that E distributions are consistent across all merged PTU files.

In [ ]:
files  = sorted(df_fret['filename'].unique())
n_files = len(files)
ncols   = min(4, n_files)
nrows   = int(np.ceil(n_files / ncols))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(4*ncols, 3*nrows),
                          sharey=True)
axes = np.array(axes).flatten()

for i, fname in enumerate(files):
    sub = df_fret[df_fret['filename'] == fname]
    axes[i].hist(sub['E_app'], bins=30, range=(0,1),
                 density=True, color='steelblue', alpha=0.7)
    axes[i].set_title(f'{fname[:30]}\n(n={len(sub)})', fontsize=7)
    axes[i].set_xlabel('E')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('E distribution per source PTU file', y=1.01)
plt.tight_layout()
plt.show()

print('Bursts per file:')
print(df_fret['filename'].value_counts().to_string())